# Taller 1: Econometría

- **Profesor:** Francisco Alfaro Medina
- **Ayudantes:** Krischnna Cortez y Karen Rojas


### Instrucciones

- Dispone de **60 minutos** para completar los **100 puntos** del taller.
- Cuide la presentación y redacción de sus respuestas.
- Puede utilizar su computador y los apuntes de clase y ayudantía.
- Debe entregar un archivo **PDF** y un **R script** (extensión `.R`).

> ⚠️ **Importante para Colab:** Este notebook usa un kernel de R. Si abre este archivo en Google Colab, seleccione **Runtime → Change runtime type → R** antes de ejecutar cualquier celda.

---
# Sección 1: Datos Aleatorios *(30 puntos)*

En esta sección trabajaremos con un dataset **simulado** de 50 alumnos de la USM. El dataset contiene las siguientes variables:

| Variable | Descripción |
|---|---|
| `hrs_sueno` | Horas de sueño promedio en el último mes |
| `profesor_part` | Si recibió ayuda de profesor particular (0/1) |
| `media_sem_pasado` | Promedio de notas del semestre anterior |
| `tiempo_est` | Horas de estudio dedicadas |
| `asistencia` | Porcentaje de asistencia a clases |
| `nivel_socioec` | Nivel socioeconómico (1 al 5) |
| `notas` | **Variable dependiente** — nota del alumno |

## Pregunta 1.1 — Generar el dataset *(6 pts.)*

Antes de ejecutar el código, **cambie la semilla** según la primera letra de su apellido:

| A–E | F–J | K–O | P–T | U–Z |
|:---:|:---:|:---:|:---:|:---:|
| 123 | 456 | 789 | 101112 | 131415 |

Reemplace el valor en `set.seed(...)` antes de continuar.

In [ ]:
# -------------------------------------------------------
# Pregunta 1.1: Generar el dataframe "datos"
# Cambie la semilla según la primera letra de su apellido
# A-E: 123 | F-J: 456 | K-O: 789 | P-T: 101112 | U-Z: 131415
# -------------------------------------------------------

set.seed(123)  # <-- CAMBIE ESTE VALOR SEGÚN SU APELLIDO
#Alvarez
datos <- data.frame(
  hrs_sueno        = round(runif(50, min = 5,  max = 10),  1),
  profesor_part    = sample(c(0, 1), 50, replace = TRUE),
  media_sem_pasado = round(runif(50, min = 60, max = 100), 1),
  tiempo_est       = round(runif(50, min = 1,  max = 8),   1),
  asistencia       = round(runif(50, min = 60, max = 100), 1),
  nivel_socioec    = sample(1:5, 50, replace = TRUE)
)

# Calcular notas con ponderaciones definidas
datos$notas <- 30 +
  datos$hrs_sueno        * 1.5  +
  datos$profesor_part    * 3    +
  datos$media_sem_pasado * 0.2  +
  datos$tiempo_est       * 2    +
  datos$asistencia       * 0.15 +
  datos$nivel_socioec    * 2    +
  rnorm(50, mean = 0, sd = 5)

# Asegurar rango entre 20 y 100
datos$notas <- pmax(pmin(datos$notas, 100), 20)

# Vista rápida del dataset
cat("Dimensiones del dataset:", nrow(datos), "filas x", ncol(datos), "columnas\n")
head(datos)

Dimensiones del dataset: 50 filas x 7 columnas


,hrs_sueno,profesor_part,media_sem_pasado,tiempo_est,asistencia,nivel_socioec,notas
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>
1,6.4,1,84.0,6.9,69.5,1,90.88856
2,8.9,0,73.3,4.5,98.5,2,80.53911
3,7.0,0,79.5,3.7,84.1,5,80.11422
4,9.4,0,98.2,2.7,80.6,5,100.00000
5,9.7,0,79.3,1.8,76.1,1,75.34071
6,5.2,1,95.6,3.7,95.2,2,87.09114


## Pregunta 1.2 — Redondear notas *(2 pts.)*

Redondee la variable `notas` a **1 decimal**.

In [ ]:
# Pregunta 1.2: Redondear notas a 1 decimal
datos$notas <- round(datos$notas, 1)

## Pregunta 1.3 — Estimadores β via álgebra matricial *(8 pts.)*

Calcule los estimadores MCO **manualmente**, usando la fórmula matricial:

$$\hat{\boldsymbol{\beta}} = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}$$

**Sin usar** la función `lm()` de R.

In [ ]:
# Pregunta 1.3: Estimadores beta via álgebra matricial (sin lm)

# Variable dependiente Y
Y <- datos$notas

# Matriz X con columna de unos para el intercepto
# Orden de variables: intercepto, hrs_sueno, profesor_part, media_sem_pasado,
#                     tiempo_est, asistencia, nivel_socioec
X <- cbind(1,
           datos$hrs_sueno,
           datos$profesor_part,
           datos$media_sem_pasado,
           datos$tiempo_est,
           datos$asistencia,
           datos$nivel_socioec)

# Fórmula de MCO: β̂ = (X'X)⁻¹ X'Y

# Paso 1: Calcular X'X (matriz de productos cruzados)
XtX <- t(X) %*% X

# Paso 2: Calcular la inversa de X'X
XtX_inv <- solve(XtX)

# Paso 3: Calcular X'Y
XtY <- t(X) %*% Y

# Paso 4: Multiplicar para obtener los betas
beta_hat_manual <- XtX_inv %*% XtY

# Mostrar resultados
cat("\n=== Betas estimados manualmente (sin usar lm() de R) ===\n")
cat("β0 (Intercepto):", round(beta_hat_manual[1], 4), "\n")
cat("β1 (hrs_sueno):", round(beta_hat_manual[2], 4), "\n")
cat("β2 (profesor_part):", round(beta_hat_manual[3], 4), "\n")
cat("β3 (media_sem_pasado):", round(beta_hat_manual[4], 4), "\n")
cat("β4 (tiempo_est):", round(beta_hat_manual[5], 4), "\n")
cat("β5 (asistencia):", round(beta_hat_manual[6], 4), "\n")
cat("β6 (nivel_socioec):", round(beta_hat_manual[7], 4), "\n")# Mostrar la matriz X'X (primeras 4x4 como ejemplo)
cat("\n=== Matriz X'X (primeras 4 filas y columnas) ===\n")
print(round(XtX[1:4, 1:4], 2))

# Mostrar el vector X'Y
cat("\n=== Vector X'Y ===\n")
print(round(XtY, 2))

# Mostrar la fórmula utilizada
cat("\n=== Nota ===\n")
cat("Los betas fueron calculados exclusivamente usando álgebra matricial:\n")
cat("β̂ = (X'X)⁻¹ X'Y\n")
cat("No se utilizó la función lm() en ningún momento del cálculo.\n")
cat("\nInterpretación de los betas:\n")
cat("β0 (Intercepto): Valor esperado de notas cuando todas las variables son 0\n")
cat("β1 (hrs_sueno): Cambio en notas por cada hora adicional de sueño\n")
cat("β2 (profesor_part): Diferencia en notas entre tener (1) o no tener (0) profesor particular\n")
cat("β3 (media_sem_pasado): Cambio en notas por cada punto en promedio del semestre pasado\n")
cat("β4 (tiempo_est): Cambio en notas por cada hora adicional de estudio\n")
cat("β5 (asistencia): Cambio en notas por cada punto porcentual de asistencia\n")
cat("β6 (nivel_socioec): Cambio en notas por cada nivel socioeconómico adicional\n")


=== Betas estimados manualmente (sin usar lm() de R) ===
β0 (Intercepto): 33.3691 
β1 (hrs_sueno): 1.1951 
β2 (profesor_part): 4.9124 
β3 (media_sem_pasado): 0.2019 
β4 (tiempo_est): 1.001 
β5 (asistencia): 0.1803 
β6 (nivel_socioec): 2.0867 

=== Matriz X'X (primeras 4 filas y columnas) ===
       [,1]     [,2] [,3]      [,4]
[1,]   50.0   380.00   23   4030.90
[2,]  380.0  2994.96  170  30655.98
[3,]   23.0   170.00   23   1867.00
[4,] 4030.9 30655.98 1867 331745.39

=== Vector X'Y ===
          [,1]
[1,]   4301.50
[2,]  32777.17
[3,]   2017.70
[4,] 348256.15
[5,]  19885.06
[6,] 347031.61
[7,]  12468.30

=== Nota ===
Los betas fueron calculados exclusivamente usando álgebra matricial:
β̂ = (X'X)⁻¹ X'Y
No se utilizó la función lm() en ningún momento del cálculo.

Interpretación de los betas:
β0 (Intercepto): Valor esperado de notas cuando todas las variables son 0
β1 (hrs_sueno): Cambio en notas por cada hora adicional de sueño
β2 (profesor_part): Diferencia en notas entre tener (1) 

## Pregunta 1.4 — Modelo con `lm()` e interpretación *(8 pts.)*

Genere el modelo de regresión múltiple usando la función `lm()` y obtenga el resumen con `summary()`.  
Luego, **interprete cada coeficiente** en el espacio indicado.

> 💡 **Tip:** Los β de `lm()` deben coincidir con los calculados manualmente en la pregunta anterior.

In [ ]:
# Pregunta 1.4: Modelo con lm() y summary
modelo_r <- lm(notas ~ hrs_sueno + profesor_part + media_sem_pasado +
               tiempo_est + asistencia + nivel_socioec, data = datos)

# Generar el resumen del modelo
cat("\n=== Resumen del modelo de regresión múltiple ===\n")
summary(modelo_r)

# Interpretación detallada de cada beta
cat("\n=== Interpretación de los betas del modelo ===\n")
cat("\n1. β0 (Intercepto):", round(coef(modelo_r)[1], 4))
cat("\n   → Cuando todas las variables independientes son cero (hrs_sueno=0, profesor_part=0,\n")
cat("     media_sem_pasado=0, tiempo_est=0, asistencia=0, nivel_socioec=0), la nota esperada\n")
cat("     es de", round(coef(modelo_r)[1], 4), "puntos. Este valor no tiene una interpretación\n")
cat("     práctica realista porque es imposible tener horas de sueño o asistencia en cero.\n")

cat("\n2. β1 (hrs_sueno):", round(coef(modelo_r)[2], 4))
cat("\n   → Por cada hora adicional de sueño promedio en el último mes, la nota del alumno\n")
cat("     aumenta en", round(coef(modelo_r)[2], 4), "puntos, manteniendo constantes las demás variables.\n")
cat("     Esto tiene sentido porque un buen descanso mejora el rendimiento académico.\n")

cat("\n3. β2 (profesor_part):", round(coef(modelo_r)[3], 4))
cat("\n   → Los alumnos que tienen profesor particular (profesor_part=1) tienen en promedio\n")
cat("     una nota", round(coef(modelo_r)[3], 4), "puntos más alta que aquellos que no tienen\n")
cat("     profesor particular (profesor_part=0), manteniendo constantes las demás variables.\n")
cat("     Esto indica que la ayuda externa contribuye positivamente al rendimiento.\n")

cat("\n4. β3 (media_sem_pasado):", round(coef(modelo_r)[4], 4))
cat("\n   → Por cada punto adicional en el promedio de notas del semestre pasado, la nota\n")
cat("     actual aumenta en", round(coef(modelo_r)[4], 4), "puntos, manteniendo constantes las demás variables.\n")
cat("     Esto refleja la consistencia en el rendimiento académico de los estudiantes.\n")

cat("\n5. β4 (tiempo_est):", round(coef(modelo_r)[5], 4))
cat("\n   → Por cada hora adicional de estudio que dedica el alumno, la nota aumenta\n")
cat("     en", round(coef(modelo_r)[5], 4), "puntos, manteniendo constantes las demás variables.\n")
cat("     Este resultado confirma la relación positiva entre tiempo de estudio y rendimiento.\n")

cat("\n6. β5 (asistencia):", round(coef(modelo_r)[6], 4))
cat("\n   → Por cada punto porcentual adicional de asistencia a clases, la nota del alumno\n")
cat("     aumenta en", round(coef(modelo_r)[6], 4), "puntos, manteniendo constantes las demás variables.\n")
cat("     La asistencia regular a clases facilita el aprendizaje y mejora las calificaciones.\n")

cat("\n7. β6 (nivel_socioec):", round(coef(modelo_r)[7], 4))
cat("\n   → Por cada nivel socioeconómico adicional (en una escala del 1 al 5), la nota\n")
cat("     aumenta en", round(coef(modelo_r)[7], 4), "puntos, manteniendo constantes las demás variables.\n")
cat("     Esto sugiere que los estudiantes con mayor nivel socioeconómico tienden a tener\n")
cat("     mejores resultados académicos, posiblemente debido a mayores recursos y oportunidades.\n")

# Información adicional del modelo
cat("\n=== Información adicional del modelo ===\n")
cat("R²:", round(summary(modelo_r)$r.squared, 4))
cat("\nR² Ajustado:", round(summary(modelo_r)$adj.r.squared, 4))
cat("\nEstadístico F:", round(summary(modelo_r)$fstatistic[1], 2))
cat("\nValor p del modelo:", format.pval(pf(summary(modelo_r)$fstatistic[1],
                                           summary(modelo_r)$fstatistic[2],
                                           summary(modelo_r)$fstatistic[3],
                                           lower.tail = FALSE), digits = 4))

cat("\n\n=== Conclusión general ===\n")
cat("Todas las variables independientes tienen el signo esperado (positivo), indicando que\n")
cat("contribuyen positivamente al rendimiento académico. Los coeficientes estimados representan\n")
cat("el efecto marginal de cada variable sobre la nota final, manteniendo constantes las demás.\n")
cat("El modelo es estadísticamente significativo (p-value < 0.05), lo que indica que en conjunto\n")
cat("las variables explican adecuadamente la variabilidad en las notas de los estudiantes.\n")



=== Resumen del modelo de regresión múltiple ===



Call:
lm(formula = notas ~ hrs_sueno + profesor_part + media_sem_pasado + 
    tiempo_est + asistencia + nivel_socioec, data = datos)

Residuals:
    Min      1Q  Median      3Q     Max 
-7.5651 -3.4436  0.2434  2.7986  8.7644 

Coefficients:
                 Estimate Std. Error t value Pr(>|t|)    
(Intercept)      33.36907    8.50166   3.925 0.000308 ***
hrs_sueno         1.19512    0.44030   2.714 0.009519 ** 
profesor_part     4.91236    1.37464   3.574 0.000884 ***
media_sem_pasado  0.20193    0.05667   3.563 0.000912 ***
tiempo_est        1.00101    0.41041   2.439 0.018926 *  
asistencia        0.18031    0.06077   2.967 0.004895 ** 
nivel_socioec     2.08675    0.43137   4.837 1.72e-05 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 4.478 on 43 degrees of freedom
Multiple R-squared:  0.6161,	Adjusted R-squared:  0.5626 
F-statistic:  11.5 on 6 and 43 DF,  p-value: 1.217e-07



=== Interpretación de los betas del modelo ===

1. β0 (Intercepto): 33.3691
   → Cuando todas las variables independientes son cero (hrs_sueno=0, profesor_part=0,
     media_sem_pasado=0, tiempo_est=0, asistencia=0, nivel_socioec=0), la nota esperada
     es de 33.3691 puntos. Este valor no tiene una interpretación
     práctica realista porque es imposible tener horas de sueño o asistencia en cero.

2. β1 (hrs_sueno): 1.1951
   → Por cada hora adicional de sueño promedio en el último mes, la nota del alumno
     aumenta en 1.1951 puntos, manteniendo constantes las demás variables.
     Esto tiene sentido porque un buen descanso mejora el rendimiento académico.

3. β2 (profesor_part): 4.9124
   → Los alumnos que tienen profesor particular (profesor_part=1) tienen en promedio
     una nota 4.9124 puntos más alta que aquellos que no tienen
     profesor particular (profesor_part=0), manteniendo constantes las demás variables.
     Esto indica que la ayuda externa contribuye positivament

## Pregunta 1.5 — Relación entre betas y ponderaciones del código *(6 pts.)*

Analice la relación entre los coeficientes estimados (β̂) y las ponderaciones reales usadas en el código del Anexo para generar las notas.



In [ ]:
# Pregunta 1.5: Comparación entre betas estimados y ponderaciones reales


# Sección 2: Wooldridge *(70 puntos)*

Para esta sección utilizaremos el paquete `wooldridge`, que contiene bases de datos clásicas de econometría.

In [ ]:
# Instalar y cargar el paquete wooldridge (solo necesario la primera vez en Colab)
if (!require(wooldridge)) install.packages("wooldridge")
library(wooldridge)
names(wage1)
head(wage1)
dim(wage1)

Loading required package: wooldridge

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘wooldridge’”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



[1] "wage"     "educ"     "exper"    "tenure"   "nonwhite" "female"  
 [7] "married"  "numdep"   "smsa"     "northcen" "south"    "west"    
[13] "construc" "ndurman"  "trcommpu" "trade"    "services" "profserv"
[19] "profocc"  "clerocc"  "servocc"  "lwage"    "expersq"  "tenursq"

,wage,educ,exper,tenure,nonwhite,female,married,numdep,smsa,northcen,⋯,trcommpu,trade,services,profserv,profocc,clerocc,servocc,lwage,expersq,tenursq
,<dbl>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<dbl>,<int>,<int>
1,3.10,11,2,0,0,1,0,2,1,0,⋯,0,0,0,0,0,0,0,1.131402,4,0
2,3.24,12,22,2,0,1,1,3,1,0,⋯,0,0,1,0,0,0,1,1.175573,484,4
3,3.00,11,2,0,0,0,0,2,0,0,⋯,0,1,0,0,0,0,0,1.098612,4,0
4,6.00,8,44,28,0,0,1,0,1,0,⋯,0,0,0,0,0,1,0,1.791759,1936,784
5,5.30,12,7,2,0,0,1,1,0,0,⋯,0,0,0,0,0,0,0,1.667707,49,4
6,8.75,16,9,8,0,0,1,0,1,0,⋯,0,0,0,1,1,0,0,2.169054,81,64


[1] 526  24

---
## 2A. Base `wage1` *(30 puntos)*

El modelo de regresión poblacional a estimar es:

$$wage = \beta_0 + \beta_1\, educ + \beta_2\, exper + \beta_3\, tenure + u$$

Donde:
- `wage` = salario por hora (USD)
- `educ` = años de escolaridad
- `exper` = años de experiencia laboral
- `tenure` = años en el trabajo actual

In [ ]:
# Cargar y limpiar la base wage1
data("wage1")
wage1 <- na.omit(wage1)

### Pregunta 2A.1 — Estimadores MCO e interpretación *(8 pts.)*

Estime el modelo completo e interprete los resultados.

In [ ]:
# Pregunta 2A.1: Modelo de regresión múltiple con wage1
modelo <- lm(wage ~ educ + exper + tenure, data = wage1)
summary(modelo)


Call:
lm(formula = wage ~ educ + exper + tenure, data = wage1)

Residuals:
    Min      1Q  Median      3Q     Max 
-7.6068 -1.7747 -0.6279  1.1969 14.6536 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept) -2.87273    0.72896  -3.941 9.22e-05 ***
educ         0.59897    0.05128  11.679  < 2e-16 ***
exper        0.02234    0.01206   1.853   0.0645 .  
tenure       0.16927    0.02164   7.820 2.93e-14 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 3.084 on 522 degrees of freedom
Multiple R-squared:  0.3064,	Adjusted R-squared:  0.3024 
F-statistic: 76.87 on 3 and 522 DF,  p-value: < 2.2e-16


### Pregunta 2A.2 — ¿Los signos son los esperados? *(10 pts.)*

Antes de ver los resultados, reflexione: ¿qué signo debería tener cada coeficiente económicamente?

In [ ]:
# Pregunta 2A.2: Revisar signos de los coeficientes
Se espera que todos los coeficientes sean positivos, ya que mayor educación, experiencia y antigüedad deberían aumentar el salario.

### Pregunta 2A.3 — Comparar R² y R² ajustado *(12 pts.)*

Estime un segundo modelo usando solo `educ` y `tenure`, y compare el ajuste con el modelo completo.

In [ ]:
# Pregunta 2A.3: Modelo reducido (sin exper)
modelo_reducido <- lm(wage ~ educ + tenure, data = wage1)

summary(modelo)$r.squared
summary(modelo)$adj.r.squared

summary(modelo_reducido)$r.squared
summary(modelo_reducido)$adj.r.squared

[1] 0.3064224

[1] 0.3024364

[1] 0.301861

[1] 0.2991912

---
## 2B. Base `attend` *(40 puntos)*

En esta sección analizamos los determinantes del rendimiento en el examen final de un curso universitario.

### Pregunta 2B.1 — Cargar la base `attend` *(4 pts.)*

In [1]:
# Pregunta 2B.1: Cargar base attend
data("attend")
attend <- na.omit(attend)

Warning message in data("attend"):
“data set ‘attend’ not found”


ERROR: Error: object 'attend' not found


### Pregunta 2B.2 — Selección de variables *(4 pts.)*

Subseleccione las siguientes variables:

| Variable | Descripción |
|---|---|
| `attend` | Clases asistidas de un total de 32 |
| `termGPA` | Promedio de notas durante el período |
| `priGPA` | Promedio acumulado antes del período |
| `ACT` | Puntaje en el examen ACT |
| `final` | **Variable dependiente** — puntaje del examen final |
| `hwrte` | Porcentaje de tareas entregadas |
| `frosh` | =1 si es estudiante de primer año |
| `soph` | =1 si es estudiante de segundo año |

In [ ]:
# Pregunta 2B.2: Subselección de variables
attend_new <- attend[, c(
  "final",
  "attend",
  "termGPA",
  "priGPA",
  "ACT",
  "hwrte",
  "frosh",
  "soph"
)]
head(attend_new)

,final,attend,termGPA,priGPA,ACT,hwrte,frosh,soph
,<int>,<int>,<dbl>,<dbl>,<int>,<dbl>,<int>,<int>
1,28,27,3.19,2.64,23,100.0,0,1
2,26,22,2.73,3.52,25,87.5,0,0
3,30,30,3.00,2.46,24,87.5,0,0
4,27,31,2.04,2.61,20,100.0,0,1
5,34,32,3.68,3.32,23,100.0,0,1
6,25,29,3.23,2.93,26,100.0,0,1


### Pregunta 2B.3 — Modelo de regresión múltiple completo *(10 pts.)*

Estime un modelo donde la variable dependiente es `final` y las independientes son todas las demás variables del subconjunto.

In [ ]:
# Pregunta 2B.3: Modelo completo con attend_new

modelo_attend <- lm(final ~ ., data = attend_new)
summary(modelo_attend)


Call:
lm(formula = final ~ ., data = attend_new)

Residuals:
     Min       1Q   Median       3Q      Max 
-15.9573  -2.5968  -0.0042   2.6582  11.0815 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept) 14.27176    1.45667   9.798  < 2e-16 ***
attend      -0.06956    0.04105  -1.694   0.0907 .  
termGPA      3.54151    0.31442  11.264  < 2e-16 ***
priGPA      -0.04149    0.40302  -0.103   0.9180    
ACT          0.27313    0.05031   5.429 7.94e-08 ***
hwrte       -0.01394    0.01043  -1.337   0.1817    
frosh       -0.61310    0.47672  -1.286   0.1989    
soph        -0.82251    0.39459  -2.084   0.0375 *  
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 3.867 on 666 degrees of freedom
Multiple R-squared:  0.3373,	Adjusted R-squared:  0.3303 
F-statistic: 48.42 on 7 and 666 DF,  p-value: < 2.2e-16


### Pregunta 2B.4 — Bondad de ajuste *(6 pts.)*

Interprete el **R²** y el **R² ajustado** del modelo anterior.

In [ ]:
# Pregunta 2B.4: Extraer métricas de bondad de ajuste
summary(modelo_attend)$r.squared
summary(modelo_attend)$adj.r.squared

El R² indica que aproximadamente el X% de la variabilidad en la nota final (final) es explicada por las variables independientes incluidas en el modelo.
El R² ajustado, que penaliza por la cantidad de variables incluidas, es ligeramente menor (Y%), lo que sugiere que el modelo mantiene un buen poder
explicativo sin incluir demasiadas variables irrelevantes.

[1] 0.3372808

[1] 0.3303152

### Pregunta 2B.5 — Modelo reducido (excluir variables no significativas) *(10 pts.)*

Genere un nuevo modelo excluyendo las variables con **p-value > 0.05** en el modelo anterior.

In [ ]:
# Identificar variables significativas (p-value <= 0.05)
summary(modelo_attend)
modelo_reducido_attend <- lm(final ~ termGPA + ACT + soph, data = attend_new)
summary(modelo_reducido_attend)


Call:
lm(formula = final ~ ., data = attend_new)

Residuals:
     Min       1Q   Median       3Q      Max 
-15.9573  -2.5968  -0.0042   2.6582  11.0815 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept) 14.27176    1.45667   9.798  < 2e-16 ***
attend      -0.06956    0.04105  -1.694   0.0907 .  
termGPA      3.54151    0.31442  11.264  < 2e-16 ***
priGPA      -0.04149    0.40302  -0.103   0.9180    
ACT          0.27313    0.05031   5.429 7.94e-08 ***
hwrte       -0.01394    0.01043  -1.337   0.1817    
frosh       -0.61310    0.47672  -1.286   0.1989    
soph        -0.82251    0.39459  -2.084   0.0375 *  
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 3.867 on 666 degrees of freedom
Multiple R-squared:  0.3373,	Adjusted R-squared:  0.3303 
F-statistic: 48.42 on 7 and 666 DF,  p-value: < 2.2e-16



Call:
lm(formula = final ~ termGPA + ACT + soph, data = attend_new)

Residuals:
     Min       1Q   Median       3Q      Max 
-14.9241  -2.5933   0.0257   2.6399  10.8382 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept) 10.93893    1.01986  10.726  < 2e-16 ***
termGPA      3.00981    0.21560  13.960  < 2e-16 ***
ACT          0.32861    0.04489   7.320 7.14e-13 ***
soph        -0.52380    0.30666  -1.708   0.0881 .  
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 3.885 on 670 degrees of freedom
Multiple R-squared:  0.3269,	Adjusted R-squared:  0.3239 
F-statistic: 108.5 on 3 and 670 DF,  p-value: < 2.2e-16


In [ ]:
# Pregunta 2B.5: Modelo reducido (solo variables significativas)
# Variables significativas identificadas: termGPA, ACT, soph
# (ajuste según los resultados de su modelo)
Se estimó un modelo reducido considerando únicamente las variables estadísticamente significativas (p ≤ 0.05) en el modelo completo.
En este caso, las variables seleccionadas fueron termGPA, ACT y soph, ya que presentan un efecto significativo sobre el rendimiento en el examen final.

Al excluir las variables no significativas, se obtiene un modelo más parsimonioso sin perder capacidad explicativa relevante.

### Pregunta 2B.6 — Comparación de modelos *(6 pts.)*

Compare el modelo completo y el modelo reducido en términos de **R² ajustado**.

In [ ]:
# Pregunta 2B.6: Comparación final de modelos

summary(modelo_attend)$adj.r.squared
summary(modelo_reducido_attend)$adj.r.squared

[1] 0.3303152

[1] 0.3239015